# Neural Prototyping

# Google Colab Mounting

In [57]:
!rm -rf /content/credit-risk-modeling
!git clone https://github.com/BillyBrothers/credit-risk-modeling.git
!pip install -r /content/credit-risk-modeling/requirements.txt

import sys 
sys.path.append('/content/credit-risk-modeling')

Cloning into 'credit-risk-modeling'...
remote: Enumerating objects: 1201, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 1201 (delta 53), reused 51 (delta 26), pack-reused 1121 (from 1)
Receiving objects: 100% (1201/1201), 52.28 MiB | 21.48 MiB/s, done.
Resolving deltas: 100% (815/815), done.
Updating files: 100% (68/68), done.


In [58]:
!pip install scikeras

In [59]:
pip install -q -U keras-tuner

In [60]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno

# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay
from credit_risk_modeling import model_eval

import tensorflow as tf
from tensorflow import keras
from keras import layers
from scikeras.wrappers import KerasClassifier
import keras_tuner as kt

In [61]:
!ls

credit-risk-modeling  logs  sample_data  untitled_project


In [62]:
!ls credit-risk-modeling

credit_risk_modeling  LICENSE	 pyproject.toml  requirements.txt
data		      Makefile	 README.md	 tests
docs		      models	 references
environment.yml       notebooks  reports


# Imports

In [63]:
X_train = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_train_neural.csv"
)

In [64]:
X_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_test_neural.csv"
)

In [65]:
X_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_val_neural.csv"
)

In [66]:
y_train= pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_train.csv"
)
y_train = y_train.values.ravel()
neg, pos = np.bincount(y_train)
total = neg + pos
print(f"Examples:\n The training sets total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The training sets total amount of samples: 22686
 Positive: 4962 (21.87% of total)


In [67]:
y_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_test.csv"
)
y_test = y_test.values.ravel()
neg, pos = np.bincount(y_test)
total = neg + pos
print(f"Examples:\n The testing set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The testing set total amount of samples: 2917
 Positive: 638 (21.87% of total)


In [68]:
y_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_val.csv"
)
y_val = y_val.values.ravel()
neg, pos = np.bincount(y_val)
total = neg + pos
print(f"Examples:\n The validation set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The validation set total amount of samples: 6806
 Positive: 1488 (21.86% of total)


# Build Sequential Models

Includes hyperparameter tuning

In [69]:
mlp_models = []
mlp_results = []

In [70]:
# # Architecture 1: Single hidden dense layer
def mlp1():
    model = keras.Sequential(name='MLP-1')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [71]:
# def mlp1(hp):
#     model1 = keras.Sequential(name='MLP-1')
#     model1.add(keras.Input(shape=(X_train.shape[1], )))

#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     hp_lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

#     model1.add(layers.Dense(units=hp_units, activation='relu'))
#     model1.add(layers.Dropout(rate= 0.20))
#     model1.add(layers.Dense(units=1,activation='sigmoid'))
    
#     model1.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     ) 
#     return model1

In [72]:
# Architecture 2: Two hidden dense layers
def mlp2():
    model = keras.Sequential(name='MLP-2')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    
    model.compile(optimizer=keras.optimizers.Adam(),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    )
    return model

In [73]:
# # Architecture 2: Two hidden dense layers
# def mlp2(hp):
#     model2 = keras.Sequential(name='MLP-2')
#     model2.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate=0.20)),
#     model2.add(layers.Dense(units=1,activation='sigmoid')),
#     model2.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )
#     return model2

In [74]:
# Architecture 3: Three hidden dense layers
def mlp3():
    model = keras.Sequential(name='MLP-3')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=256, activation='relu')),
    model.add(layers.Dropout(rate=0.2)),
    model.add(layers.Dense(units=1,activation='sigmoid')),

    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [75]:
# def mlp3(hp):
#     model3 = keras.Sequential(name='MLP-3')
#     model3.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.2)),
#     model3.add(layers.Dense(units=1,activation='sigmoid')),
#     model3.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )

#     return model3

In [76]:
mlp_models = [
    ("MLP-1", mlp1()),
    ("MLP-2", mlp2()),
    ("MLP-3", mlp3())
]

### Callbacks

In [77]:
reduce_lr_plateau = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    verbose=1,
    min_lr=0.001
)

In [78]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=7,
    verbose=1,
    restore_best_weights=True
)

In [79]:
log_dir = "logs/fit/"
tensorboard = keras.callbacks.TensorBoard(
    log_dir= log_dir
)

### Fit

In [80]:
class_weight = {
    0: 1.0,
    1: 2.0
    }

In [81]:
histories = {}

In [82]:
for model_name, model in mlp_models:
    print(f"Currently fitting model {model_name}.")
    history = model.fit(
        x= X_train,
        y= y_train,
        batch_size=32,
        epochs= 100,
        verbose=2,
        callbacks= [early_stopping, reduce_lr_plateau, tensorboard],
        validation_data= (X_val, y_val),
        class_weight= class_weight
    )
    histories[model_name] = history.history

Currently fitting model MLP-1.
Epoch 1/100
709/709 - 3s - 4ms/step - auc: 0.8505 - loss: 0.5546 - val_auc: 0.8902 - val_loss: 0.3502 - learning_rate: 1.0000e-03
Epoch 2/100
709/709 - 2s - 2ms/step - auc: 0.8883 - loss: 0.4805 - val_auc: 0.9013 - val_loss: 0.3294 - learning_rate: 1.0000e-03
Epoch 3/100
709/709 - 2s - 3ms/step - auc: 0.8946 - loss: 0.4635 - val_auc: 0.9040 - val_loss: 0.3127 - learning_rate: 1.0000e-03
Epoch 4/100
709/709 - 2s - 3ms/step - auc: 0.8999 - loss: 0.4509 - val_auc: 0.9075 - val_loss: 0.3140 - learning_rate: 1.0000e-03
Epoch 5/100
709/709 - 2s - 2ms/step - auc: 0.9018 - loss: 0.4446 - val_auc: 0.9104 - val_loss: 0.2993 - learning_rate: 1.0000e-03
Epoch 6/100
709/709 - 2s - 2ms/step - auc: 0.9029 - loss: 0.4400 - val_auc: 0.9099 - val_loss: 0.2976 - learning_rate: 1.0000e-03
Epoch 7/100
709/709 - 2s - 2ms/step - auc: 0.9065 - loss: 0.4299 - val_auc: 0.9124 - val_loss: 0.2959 - learning_rate: 1.0000e-03
Epoch 8/100
709/709 - 2s - 2ms/step - auc: 0.9080 - loss: 0

In [89]:
histories['MLP-3']['val_auc']

[0.9036178588867188,
 0.9139466285705566,
 0.918378472328186,
 0.9199810028076172,
 0.9206768870353699,
 0.9218349456787109,
 0.9234232902526855,
 0.9228512644767761,
 0.9231187105178833,
 0.921812891960144,
 0.9234161376953125,
 0.9246427416801453,
 0.9236798286437988,
 0.9243573546409607,
 0.9244530200958252,
 0.9240238070487976,
 0.9247168898582458,
 0.9250455498695374,
 0.9250538945198059,
 0.9249132871627808,
 0.9244586229324341,
 0.9243122339248657,
 0.9243274331092834,
 0.9213683605194092,
 0.9224716424942017,
 0.9221221208572388,
 0.9239667654037476,
 0.9246728420257568,
 0.9243037104606628,
 0.925274670124054,
 0.9244345426559448]

In [160]:
max_auc_dict = dict([sorted(max_auc_per_model.items(), key= lambda item: item[1])[-1]])

In [164]:
list(max_auc_dict.keys())[0]

'MLP-2'

In [174]:
chosen_metric = 'val_auc'
max_auc_per_model = {}
best_auc = None
best_model = None
best_model_name = None

for model_name, model in mlp_models:
    max_auc_per_model[model_name] = max(histories[model_name][chosen_metric])
    max_auc_dict = dict([sorted(max_auc_per_model.items(), key= lambda item: item[1])[-1]])
    max_auc_model_name = list(max_auc_dict.keys())[0]
    max_aux_score = list(max_auc_dict.values())[0]
    if max_auc_model_name == model_name:
        best_auc = max_aux_score
        best_model_name = max_auc_model_name
        best_model = model
    else:
        continue

print(f"Results: \nBest model: {best_model_name}\nAUC score: {best_auc:.4f}")

Results: 
Best model: MLP-2
AUC score: 0.9262


In [ ]:
# tuner = kt.Hyperband(
#     hypermodel= mlp1,
#     objective="val_auc",
#     max_epochs= 100,
#     factor=3,
# )

In [ ]:
# for tid, t in tuner.oracle.trials.items():
#     print(tid, t.status, t.score)


In [ ]:
# tuner.search_space_summary()

In [ ]:
# tuner.search(
#     X_train,
#     y_train,
#     validation_data= (X_val, y_val),
#     callbacks = [early_stopping]
# )

In [ ]:
# tuner.results_summary()

### Inference

In [ ]:
models_only = []

In [ ]:
for model_name, model in mlp_models:
    model = KerasClassifier(model)
    models_only.append(model)

In [ ]:
models_only

[KerasClassifier(
 	model=<Sequential name=MLP-64, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_weight=None
 ),
 KerasClassifier(
 	model=<Sequential name=MLP-64-128, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_weight=None
 ),
 KerasClassifier(
 	model=<Sequential name=MLP-64-128-256, built=True>
 	build_fn=None
 	warm_start=False
 	random_state=None
 	optimizer=rmsprop
 	loss=None
 	metrics=None
 	batch_size=None
 	validation_batch_size=None
 	verbose=1
 	callbacks=None
 	validation_split=0.0
 	shuffle=True
 	run_eagerly=False
 	epochs=1
 	class_we

In [ ]:
untuned_model_performance, fitted_models = model_eval.comparing_models(
    models= models_only,
    X_train= X_train,
    y_train= y_train,
    X_test= X_test,
    y_test= y_test
)

709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - auc: 0.9218 - loss: 0.2466
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - auc: 0.9217 - loss: 0.2349
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
709/709 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - auc: 0.9211 - loss: 0.2367
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
